# MATH 60231 – Homework #4
**Group:** Mathis Huot, Nathan Madore, Isabelle Tanara

Due: Monday, November 30, 2026

# Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from pathlib import Path
from io import BytesIO, StringIO
from urllib.request import Request, urlopen
from zipfile import ZipFile

## Table formatting functions

Tables are drawn in the style of finance research papers (horizontal rules only, serif font, optional panels and notes). The functions are reused from our own Case Study 1 notebook. The underlying matplotlib table/text drawing follows [MatPlotLib - Table Demo](https://matplotlib.org/stable/gallery/misc/table_demo.html).

In [2]:
# Table formatting (reused from our Case Study 1 notebook, adapted to one Output/ folder)
import os, re, textwrap
from IPython.display import display
os.makedirs("Output", exist_ok=True)                       # every table and figure is also saved as a PNG in Output/


def save_output(fig, label):
    """Save a matplotlib figure as a PNG (200 dpi) in the Output/ folder.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to save.
    label : str
        Table/figure title; turned into a safe file name
        (e.g. "Table 2.1 - Descriptive statistics" -> "Output/Table_2.1_Descriptive_statistics.png").
    """
    name = re.sub(r"[^A-Za-z0-9.]+", "_", label).strip("_.")[:90]
    fig.savefig(f"Output/{name}.png", dpi=200, bbox_inches="tight")


# ---------- Number formats used in the tables ----------
#   "num"  -> 2 decimals            "num3" -> 3 decimals (e.g. autocorrelations)
#   "num4" -> 4 decimals            "pval" -> 3 decimals (p-values)
#   "pct"  -> 0.0013 shown as 0.13% "int"  -> 397

def format_value(x, kind="num"):
    """Format one number as text for a table.

    Parameters
    ----------
    x : float or str
        Value to format (NaN is shown as a dash, text such as a date is shown as is).
    kind : str
        "num" (2 decimals), "num3" (3 decimals), "num4" (4 decimals), "pval" (3 decimals),
        "pct" (percentage, 2 decimals) or "int" (whole number).

    Returns
    -------
    str
        Formatted value, e.g. 0.0013 with kind="pct" -> "0.13%".
    """
    if isinstance(x, str):
        return x
    if pd.isna(x):
        return "–"
    if kind == "pct":
        return f"{x * 100:.2f}%"
    if kind == "int":
        return f"{x:,.0f}"
    if kind in ("pval", "num3"):
        return f"{x:,.3f}"
    if kind == "num4":
        return f"{x:,.4f}"
    return f"{x:,.2f}"


def format_table(df, row_formats=None, col_formats=None, default="num"):
    """Return a copy of a table with every number formatted as text.

    The format of a cell is taken from its column (col_formats) if listed, otherwise from
    its row (row_formats), otherwise `default`.

    Parameters
    ----------
    df : pd.DataFrame
        Table of numbers.
    row_formats : dict, optional
        Maps a row label to a format kind (see format_value).
    col_formats : dict, optional
        Maps a column label to a format kind (useful when rows are series and columns are statistics).
    default : str
        Format kind used when neither the row nor the column is listed.

    Returns
    -------
    pd.DataFrame
        Same shape as df, with formatted strings.
    """
    row_formats, col_formats = row_formats or {}, col_formats or {}
    out = df.astype(object).copy()
    for r in df.index:
        for c in df.columns:
            kind = col_formats.get(c, row_formats.get(r, default))
            out.loc[r, c] = format_value(df.loc[r, c], kind)
    return out


def split_title(title):
    """Split "Table 2.1 - Description" (hyphen or en dash) into ("Table 2.1", "Description")."""
    number, _, description = re.sub(r"\s+[–-]\s+", " - ", title, count=1).partition(" - ")
    return number, description


# ---------- Table image in the style of finance research papers ----------
def save_table_png(panels, title, notes=None, row_formats=None, col_formats=None, default="num",
                   width=7.5, label_width=3.3):
    """Draw a table as an image in the style of finance research papers and save it to Output/.

    Horizontal rules only (thick top and bottom, thin under the header and between panels),
    italic panel labels, serif font, and an optional notes paragraph under the table.

    Parameters
    ----------
    panels : dict
        {panel label: DataFrame}; all panels must have the same columns ("" = no panel label).
    title : str
        "Table X.Y - Description" (number in bold, description below).
    notes : str, optional
        Notes printed under the table.
    row_formats, col_formats, default
        Number formats (see format_table).
    width : float
        Width of the image in inches.
    label_width : float
        Width of the row-label column in inches.
    """
    columns = list(next(iter(panels.values())).columns)

    # List of lines to draw, top to bottom
    lines = []
    for panel_name, df in panels.items():
        if panel_name:
            lines.append(("panel", panel_name))
        for label, row in format_table(df, row_formats, col_formats, default).iterrows():
            lines.append(("row", label, list(row)))

    # Sizes in inches (the drawing uses inches as coordinates, measured from the top)
    margin, row_h = 0.3, 0.26
    title_h = 0.65
    notes_text = textwrap.fill(notes, int(150 * width / 7.5)) if notes else ""
    notes_h = 0.16 * (notes_text.count("\n") + 1) + 0.2 if notes else 0.1
    table_h = row_h * (len(lines) + 1.6) + 0.12 * (len(panels) - 1)
    height = title_h + table_h + notes_h

    left, right = margin, width - margin
    first_value_x = margin + label_width                 # values start to the right of the row labels
    col_w = (right - first_value_x) / len(columns)
    col_x = [first_value_x + (i + 0.5) * col_w for i in range(len(columns))]

    with plt.rc_context({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"]}):
        fig = plt.figure(figsize=(width, height))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)                           # y grows downward
        ax.axis("off")

        number, description = split_title(title)
        ax.text(width / 2, 0.2, number, ha="center", va="center", fontsize=11, fontweight="bold")
        ax.text(width / 2, 0.45, description, ha="center", va="center", fontsize=11)

        y = title_h
        ax.hlines(y, left, right, color="black", linewidth=1.3)              # top rule
        y += row_h * 0.6
        for x, col in zip(col_x, columns):
            ax.text(x, y, col, ha="center", va="center", fontsize=10)
        y += row_h * 0.55
        ax.hlines(y, left, right, color="black", linewidth=0.6)              # rule under the header
        y += row_h * 0.55

        first_panel = True
        for line in lines:
            if line[0] == "panel":
                if not first_panel:                                          # thin rule between panels
                    y += 0.06
                    ax.hlines(y - row_h * 0.5, left, right, color="black", linewidth=0.4)
                    y += 0.06
                ax.text(left, y, line[1], ha="left", va="center", fontsize=10, style="italic")
                first_panel = False
            else:
                _, label, values = line
                ax.text(left + 0.15, y, label, ha="left", va="center", fontsize=10)
                for x, value in zip(col_x, values):
                    ax.text(x, y, value, ha="center", va="center", fontsize=10)
            y += row_h

        y -= row_h * 0.45
        ax.hlines(y, left, right, color="black", linewidth=1.3)              # bottom rule

        if notes:
            ax.text(left, y + 0.12, notes_text, ha="left", va="top", fontsize=8, linespacing=1.4)

        save_output(fig, title)
        plt.close(fig)   # saved only; the formatted table is displayed in the notebook instead


def save_table(table, title, notes=None, row_formats=None, col_formats=None, default="num", **layout):
    """Save a table as a finance-paper-style PNG in Output/ and display it formatted in the notebook.

    Parameters
    ----------
    table : pd.DataFrame or dict
        One table, or {"Panel A: ...": df_a, "Panel B: ...": df_b} for a multi-panel table.
    title : str
        "Table X.Y - Description"; also used for the file name.
    notes : str, optional
        Notes printed under the table.
    row_formats, col_formats, default
        Number formats (see format_table).
    **layout
        Optional width and label_width of the image (see save_table_png).
    """
    panels = table if isinstance(table, dict) else {"": table}
    save_table_png(panels, title, notes, row_formats, col_formats, default, **layout)

    print(title)
    for panel_name, df in panels.items():
        if panel_name:
            print(panel_name)
        display(format_table(df, row_formats, col_formats, default))

## Data loading

In [3]:
DATA_DIR = Path("Data")
FRENCH_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{}_CSV.zip"
FRENCH_FILES = ["F-F_Research_Data_5_Factors_2x3", "F-F_Momentum_Factor", "17_Industry_Portfolios"]
SAMPLE = ("1980-01", "2021-12")          # monthly sample used in Problems 1 and 2 (504 months)


def french_csv(name):
    """Return the text of one Ken French Data Library CSV, downloading it to Data/ the first time.

    Parameters
    ----------
    name : str
        File name in the library without the "_CSV.zip" suffix, e.g. "F-F_Momentum_Factor".

    Returns
    -------
    str
        Raw text of the CSV file.
    """
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        request = Request(FRENCH_URL.format(name), headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
            path.write_bytes(archive.read(archive.namelist()[0]))      # each zip holds a single CSV
    return path.read_text(encoding="latin-1")


# Download (first run only) and check the three raw files
for name in FRENCH_FILES:
    print(f"{name}: {len(french_csv(name).splitlines())} lines")




F-F_Research_Data_5_Factors_2x3: 831 lines
F-F_Momentum_Factor: 1316 lines
17_Industry_Portfolios: 5254 lines


# Problem 1 – Tests of Asset Pricing Models [50 pts]
Monthly data, January 1980 – December 2021, from Prof. Ken French's Data Library.

## 1a. `Q1_KenFrenchData()` [10 pts]
Function with no arguments returning `factors` (Mkt-RF, SMB, HML, RMW, CMA from *F-F_Research_Data_5_Factors_2x3*, then Mom from *F-F_Momentum_Factor*) and `portfolios` (17 Industry Portfolios). All series must be **excess returns** (net of RF). Report and interpret the first five rows and descriptive statistics of each object.

In [4]:
def french_table(text):
    """Parse the first monthly table of a Ken French CSV.

    Each file stacks several tables (monthly, then annual, ...) separated by text lines.
    The monthly table is the first block of rows whose first field is a YYYYMM date.

    Parameters
    ----------
    text : str
        Raw text of the CSV (see french_csv).

    Returns
    -------
    title : str
        Text line above the table (e.g. "Average Value Weighted Returns -- Monthly"), "" if none.
    df : pd.DataFrame
        Monthly returns in percent, indexed by a monthly PeriodIndex; missing codes (-99.99, -999) set to NaN.
    """
    # Build dataframe strcuture for data insertion
    is_row = lambda line: re.match(r"\s*\d{6}\s*,", line) is not None
    lines = text.splitlines()
    start = next(i for i, line in enumerate(lines) if is_row(line))           
    end = next((i for i in range(start, len(lines)) if not is_row(lines[i])), len(lines))
    title = next((line.strip() for line in reversed(lines[:start - 1]) if line.strip()), "")

    # Add step that allows insertion of data into the newly built empty dataframe
    df = pd.read_csv(StringIO("\n".join(lines[start - 1:end])), index_col=0)  
    df.columns = df.columns.str.strip()                                          
    df.index = pd.PeriodIndex(df.index.astype(str), freq="M")
    return title, df.replace([-99.99, -999], np.nan)


def Q1_KenFrenchData():
    """Monthly factors and 17 industry portfolios, January 1980 - December 2021, as excess returns (%).

    Returns
    -------
    factors : pd.DataFrame (504 x 6)
        Mkt-RF, SMB, HML, RMW, CMA (F-F_Research_Data_5_Factors_2x3) and Mom (F-F_Momentum_Factor).
        Mkt-RF is already the market return minus RF; SMB, HML, RMW, CMA and Mom are long-short,
        zero-cost portfolios, so they are already excess returns and need no adjustment.
    portfolios : pd.DataFrame (504 x 17)
        Value-weighted returns of the 17 Industry Portfolios minus RF (the one-month T-bill rate
        reported in the 5-factor file).
    """
    # Insert the CSV data into the French_Table function built above
    _, ff5 = french_table(french_csv("F-F_Research_Data_5_Factors_2x3"))
    _, mom = french_table(french_csv("F-F_Momentum_Factor"))
    title, ind = french_table(french_csv("17_Industry_Portfolios"))
    assert "Value Weighted" in title, title       # first table of the file = value-weighted monthly returns

    # Reconstruct table to hold the columns we want
    start, end = SAMPLE
    ff5, mom, ind = (df.loc[start:end] for df in (ff5, mom, ind))
    factors = pd.concat([ff5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1)
    portfolios = ind.sub(ff5["RF"], axis=0)       # raw industry returns -> excess returns

    # Full balanced sample, no missing months
    assert len(factors) == len(portfolios) == 504
    assert factors.notna().all().all() and portfolios.notna().all().all()
    return factors, portfolios

In [5]:
# Build tables from function above
factors, portfolios = Q1_KenFrenchData()

def with_month_labels(df):
    """Copy of df with the monthly index written as "1980-01" (row labels of the tables)."""
    return df.set_axis(df.index.strftime("%Y-%m"))

# Table 1.1: factors -- first five rows (Panel A) and descriptive statistics (Panel B)
save_table({"Panel A: First five rows": with_month_labels(factors.head()),
            "Panel B: Descriptive statistics": factors.describe()},
           "Table 1.1 - Monthly factors, January 1980 - December 2021 (excess returns, %)",
           notes="Mkt-RF, SMB, HML, RMW and CMA from F-F_Research_Data_5_Factors_2x3; Mom from F-F_Momentum_Factor "
                 "(Ken French Data Library). Returns in percent per month.",
           row_formats={"count": "int"}, label_width=2.2)

# Tables 1.2 and 1.3: 17 industry portfolios (one row per industry, the 17 columns do not fit across a page)
save_table(with_month_labels(portfolios.head()).T,
           "Table 1.2 - 17 industry portfolios, first five months (excess returns, %)",
           notes="Value-weighted 17 Industry Portfolios returns minus the one-month T-bill rate (RF), Ken French Data Library.",
           label_width=1.5)
save_table(portfolios.describe().T,
           "Table 1.3 - Descriptive statistics of the 17 industry portfolios (excess returns, %)",
           notes="Monthly value-weighted returns minus RF, January 1980 - December 2021.",
           col_formats={"count": "int"}, label_width=1.5)

Table 1.1 - Monthly factors, January 1980 - December 2021 (excess returns, %)
Panel A: First five rows


,Mkt-RF,SMB,HML,RMW,CMA,Mom
1980-01,5.50,1.89,1.85,-1.70,1.93,7.46
1980-02,-1.23,-1.50,0.59,0.07,2.94,7.91
1980-03,-12.89,-6.99,-0.96,1.57,-1.07,-9.59
1980-04,3.96,1.05,1.03,-2.12,0.33,-0.50
1980-05,5.26,1.99,0.38,0.38,-0.62,-1.18


Panel B: Descriptive statistics


,Mkt-RF,SMB,HML,RMW,CMA,Mom
count,504,504,504,504,504,504
mean,0.74,0.09,0.20,0.38,0.25,0.53
std,4.48,2.90,3.03,2.40,1.97,4.48
min,-23.19,-15.53,-13.83,-18.93,-6.42,-34.36
25%,-1.91,-1.58,-1.52,-0.75,-1.02,-1.21
50%,1.18,0.01,0.07,0.32,0.08,0.63
75%,3.48,1.88,1.75,1.38,1.46,2.85
max,13.58,18.54,12.25,13.04,9.00,18.05


Table 1.2 - 17 industry portfolios, first five months (excess returns, %)


,1980-01,1980-02,1980-03,1980-04,1980-05
Food,3.06,-6.18,-10.24,5.79,7.16
Mines,13.29,2.89,-22.35,1.10,9.18
Oil,12.91,12.17,-19.10,5.60,3.37
Clths,1.97,-1.79,-10.28,2.12,2.98
Durbl,3.93,-6.18,-6.79,0.96,4.99
Chems,3.18,-2.50,-13.28,3.58,6.70
Cnsum,-3.10,-4.92,-4.01,3.80,7.26
Cnstr,7.81,-3.01,-16.08,2.77,8.06
Steel,16.78,-0.60,-19.70,1.04,4.39
FabPr,6.42,-1.91,-17.92,2.69,4.89


Table 1.3 - Descriptive statistics of the 17 industry portfolios (excess returns, %)


,count,mean,std,min,25%,50%,75%,max
Food,504,0.88,4.18,-18.77,-1.31,0.96,3.42,14.68
Mines,504,0.62,7.97,-32.88,-4.62,0.46,5.86,22.47
Oil,504,0.64,6.39,-34.78,-2.77,0.76,4.28,32.80
Clths,504,0.88,6.12,-32.11,-2.20,1.05,4.73,23.92
Durbl,504,0.61,5.72,-25.91,-2.57,0.79,4.06,29.61
Chems,504,0.75,5.85,-28.51,-2.67,1.04,4.22,21.88
Cnsum,504,0.88,4.31,-19.67,-1.79,1.09,3.59,15.39
Cnstr,504,0.92,6.03,-29.85,-2.81,1.10,4.32,18.02
Steel,504,0.50,8.18,-32.49,-4.53,0.47,5.48,29.87
FabPr,504,0.75,5.50,-29.08,-2.28,1.04,4.00,18.56


## 1b. Descriptive statistics & graphs [10 pts]
Graphs (at minimum): (i) time-series plot of the six factors; (ii) time-series plot of the 17 industry excess returns; (iii) correlation heatmap of the factors. Discuss: most volatile factors, industries with highest/lowest mean excess returns, multicollinearity implied by the factor correlations.

## 1c. `Q1_AssetPricingModel()` [20 pts]
Inputs: `Factors` (T×K), `Assets` (T×N), boolean `Traded`. Two-pass estimation: (i) time-series regressions $r_{it} = \alpha_i + \beta_i' F_t + \varepsilon_{it}$; (ii) cross-sectional regression of mean excess returns $\bar r_i$ on $\hat\beta_i$ to get risk premia $\hat\lambda$.

Prints a summary and returns: $\hat\lambda$ (K×1) and robust t-stats (K×1); $\hat\beta$ (N×K) and robust t-stats (N×K); cross-sectional $R^2$; J-test [statistic, p-value, dof] (3×1). Apply with `Traded=True` and `Traded=False`; report and interpret both.

In [ ]:
def time_series_pass(F, R):
    """Time-series pass: regress every asset on a constant and the K factors (OLS, all assets at once).

    For each asset i:  r_it = alpha_i + beta_i' F_t + e_it.
    Stacking the N regressions side by side, the OLS coefficients are B = (X'X)^-1 X'R,
    where X = [1, F] is the same regressor matrix for every asset.

    Parameters
    ----------
    F : np.ndarray (T x K)
        Factor returns.
    R : np.ndarray (T x N)
        Excess returns of the assets.

    Returns
    -------
    alpha : np.ndarray (N,)
        Intercepts (pricing errors when the factors are traded).
    beta : np.ndarray (N x K)
        Factor loadings.
    beta_t : np.ndarray (N x K)
        Heteroscedasticity-robust (White, HC0) t-statistics of the loadings.
    resid : np.ndarray (T x N)
        Residuals, used later for the J-test.
    """
    T = F.shape[0]
    X = np.column_stack([np.ones(T), F])                 # constant + factors
    XtX_inv = np.linalg.inv(X.T @ X)
    B = XtX_inv @ X.T @ R                                
    resid = R - X @ B                                     # Actual - fitted

    # White's sandwich (HC0) 
    meat = np.einsum("ta,tb,ti->iab", X, X, resid**2)     # N x (K+1) x (K+1), one "meat" matrix per asset
    cov = XtX_inv @ meat @ XtX_inv                        
    se = np.sqrt(np.diagonal(cov, axis1=1, axis2=2))      # N x (K+1) standard errors (the diagonal of the matrix)

    alpha, beta = B[0], B[1:].T
    beta_t = beta / se[:, 1:]                             # t-test
    return alpha, beta, beta_t, resid

In [ ]:
def risk_premia(F, R, beta, traded):
    """Step 2: risk premia lambda and their heteroscedasticity-robust t-statistics.

    Traded factors (excess returns on traded portfolios) must price themselves, so their premium
    is their sample mean. Otherwise lambda comes from the cross-sectional regression (no constant)
    of the N mean excess returns on the estimated loadings: r_bar_i = beta_i' lambda + a_i.

    Parameters
    ----------
    F : np.ndarray (T x K)
        Factor returns.
    R : np.ndarray (T x N)
        Excess returns of the assets.
    beta : np.ndarray (N x K)
        Loadings from the time-series pass.
    traded : bool
        True if the factors are excess returns on traded assets.

    Returns
    -------
    lam : np.ndarray (K,)
        Risk premia.
    lam_t : np.ndarray (K,)
        Heteroscedasticity-robust (White, HC0) t-statistics.
    """
    T = F.shape[0]
    if traded:
        lam = F.mean(axis=0)                                    # When traded, theory says; premium = average factor return 
        se = np.sqrt(((F - lam) ** 2).sum(axis=0)) / T          # White s.e. (sample mean)
    else:
        r_bar = R.mean(axis=0)                                  # N average excess returns
        BtB_inv = np.linalg.inv(beta.T @ beta)                  # K x K
        lam = BtB_inv @ beta.T @ r_bar                          # OLS of r_bar on beta, no constant
        a = r_bar - beta @ lam                                  # N cross-sectional pricing errors (vs if traded)
        cov = BtB_inv @ (beta.T * a**2) @ beta @ BtB_inv        # White sandwich over N assets
        se = np.sqrt(np.diag(cov))
    return lam, lam / se

In [11]:
def cross_sectional_r2(R, beta, lam):
    """Step 3: cross-sectional R^2 of the model.

    Share of the cross-sectional dispersion in average excess returns explained by beta_i' lambda:
    R^2 = 1 - sum_i a_i^2 / sum_i (r_bar_i - mean(r_bar))^2, with pricing errors a_i = r_bar_i - beta_i' lambda.
    No constant in the model, so the pricing errors also include any error in the average level,
    and R^2 can be negative (the model then fits worse than the cross-sectional mean).

    Parameters
    ----------
    R : np.ndarray (T x N)
        Excess returns of the assets.
    beta : np.ndarray (N x K)
        Loadings from the time-series pass.
    lam : np.ndarray (K,)
        Risk premia from risk_premia().

    Returns
    -------
    float
        Cross-sectional R^2.
    """
    r_bar = R.mean(axis=0)                                  # N average excess returns
    a = r_bar - beta @ lam                                  # N pricing errors
    return 1 - np.sum(a**2) / np.sum((r_bar - r_bar.mean())**2)

In [16]:
def j_test(F, R, alpha, beta, resid, lam, traded):
    """Step 4: J-test of H0 = the model explains all cross-sectional differences in expected excess returns.

    Traded=True  (Cochrane 2005, eq. 12.3): pricing errors are the time-series intercepts alpha,
        J = T * alpha' Sigma^-1 alpha / (1 + f_bar' Omega^-1 f_bar)  ~  chi2(N)
    Traded=False (Cochrane 2005, eq. 12.13, with Shanken correction): pricing errors are a = r_bar - beta lambda,
        J = a' Cov(a)^+ a  ~  chi2(N - K),  Cov(a) = (1/T) M Sigma M' (1 + lambda' Omega^-1 lambda),
        M = I - beta (beta'beta)^-1 beta'
    Sigma = residual covariance (N x N), Omega = factor covariance (K x K), ^+ = pseudo-inverse.

    Parameters
    ----------
    F : np.ndarray (T x K)
        Factor returns.
    R : np.ndarray (T x N)
        Excess returns of the assets.
    alpha, beta, resid
        Intercepts (N,), loadings (N x K) and residuals (T x N) from time_series_pass().
    lam : np.ndarray (K,)
        Risk premia from risk_premia().
    traded : bool
        True if the factors are excess returns on traded assets.

    Returns
    -------
    np.ndarray (3,)
        [J statistic, p-value, degrees of freedom].
    """
    T, K = F.shape
    N = R.shape[1]
    sigma = resid.T @ resid / T                             # N x N residual covariance
    omega = np.cov(F, rowvar=False, bias=True)              # K x K factor covariance

    if traded:
        f_bar = F.mean(axis=0)
        J = T * (alpha @ np.linalg.solve(sigma, alpha)) / (1 + f_bar @ np.linalg.solve(omega, f_bar))
        dof = N
    else:
        a = R.mean(axis=0) - beta @ lam                     # cross-sectional pricing errors
        M = np.eye(N) - beta @ np.linalg.solve(beta.T @ beta, beta.T)          
        cov_a = M @ sigma @ M.T * (1 + lam @ np.linalg.solve(omega, lam)) / T  # Shanken-corrected Cov(a)
        J = a @ np.linalg.pinv(cov_a) @ a                   
        dof = N - K

    return np.array([J, stats.chi2.sf(J, dof), dof])

In [ ]:
def Q1_AssetPricingModel(Factors, Assets, Traded):
    """Two-pass estimation and test of a linear asset pricing model.

    (i)  Time-series pass: r_it = alpha_i + beta_i' F_t + e_it for each asset (time_series_pass).
    (ii) Risk premia lambda: factor means if Traded, else cross-sectional regression of r_bar on beta (risk_premia).
    Then the cross-sectional R^2 (cross_sectional_r2) and the J-test (j_test). Prints a summary.

    Parameters
    ----------
    Factors : pd.DataFrame (T x K)
        Factor returns.
    Assets : pd.DataFrame (T x N)
        Excess returns on assets or portfolios.
    Traded : bool
        True if the factors are excess returns on traded assets.

    Returns
    -------
    lam : np.ndarray (K x 1)
        Risk premia.
    lam_t : np.ndarray (K x 1)
        Heteroscedasticity-robust t-statistics of lam.
    beta : np.ndarray (N x K)
        Factor loadings.
    beta_t : np.ndarray (N x K)
        Heteroscedasticity-robust t-statistics of beta.
    R2 : float
        Cross-sectional R^2.
    J : np.ndarray (3 x 1)
        [J statistic, p-value, degrees of freedom] for H0: all pricing errors are zero.
    """
    F, R = Factors.to_numpy(), Assets.to_numpy()

    alpha, beta, beta_t, resid = time_series_pass(F, R)          # (i) time-series pass
    lam, lam_t = risk_premia(F, R, beta, Traded)                 # (ii) risk premia
    R2 = cross_sectional_r2(R, beta, lam)
    J = j_test(F, R, alpha, beta, resid, lam, Traded)

    # Printed summary
    print(f"Linear asset pricing model, Traded={Traded}  (T={len(F)}, N={R.shape[1]}, K={F.shape[1]})")
    print(pd.DataFrame({"lambda": lam, "t-stat": lam_t}, index=Factors.columns).round(3).to_string())
    print(f"Cross-sectional R2: {R2:.3f}")
    print(f"J-test: J = {J[0]:.2f}, p-value = {J[1]:.4f}, chi2({J[2]:.0f})")
    print("Factor loadings (t-stats in parentheses):")
    loadings = (pd.DataFrame(beta, Assets.columns, Factors.columns).map("{:.2f}".format)
                + pd.DataFrame(beta_t, Assets.columns, Factors.columns).map(" ({:.1f})".format))
    print(loadings.to_string())

    return lam.reshape(-1, 1), lam_t.reshape(-1, 1), beta, beta_t, R2, J.reshape(-1, 1)

In [18]:
# Run the model under both assumptions (each call prints its own summary)
results = {traded: Q1_AssetPricingModel(factors, portfolios, Traded=traded) for traded in (True, False)}

# Table 1.4: risk premia (Panel A) and model fit / J-test (Panel B), both versions side by side
panel_a = pd.DataFrame(index=factors.columns)
panel_b = pd.DataFrame(index=["Cross-sectional R²", "J statistic", "p-value", "Degrees of freedom"])
for traded, (lam, lam_t, beta, beta_t, R2, J) in results.items():
    label = "traded" if traded else "non-traded"
    panel_a[f"λ, {label}"] = lam.ravel()
    panel_a[f"t, {label}"] = lam_t.ravel()
    panel_b[f"λ, {label}"] = [R2, *J.ravel()]
    panel_b[f"t, {label}"] = np.nan                          # no t-stat for these rows (shown as a dash)

save_table({"Panel A: Risk premia (% per month)": panel_a, "Panel B: Model fit and J-test": panel_b},
           "Table 1.4 - Two-pass estimates of the six-factor model, 17 industry portfolios",
           notes="Traded: lambda = factor means, J = T a' S^-1 a / (1 + f' W^-1 f) ~ chi2(N). Non-traded: lambda from the "
                 "cross-sectional regression of mean excess returns on betas (no constant), J with Shanken correction "
                 "~ chi2(N-K). t-statistics use White (HC0) standard errors. R² = 1 - sum(a²) / sum((r - mean r)²). "
                 "Monthly data, January 1980 - December 2021, T = 504, N = 17, K = 6.",
           row_formats={"p-value": "pval", "Degrees of freedom": "int"}, default="num3", label_width=2.0)

# Table 1.5: factor loadings (identical in both calls: the time-series pass does not depend on Traded)
_, _, beta, beta_t, _, _ = results[True]
loadings = (pd.DataFrame(beta, portfolios.columns, factors.columns).map("{:.2f}".format)
            + pd.DataFrame(beta_t, portfolios.columns, factors.columns).map(" ({:.1f})".format))
save_table(loadings, "Table 1.5 - Factor loadings of the 17 industry portfolios (time-series pass)",
           notes="OLS of each industry's monthly excess return on a constant and the six factors, January 1980 - "
                 "December 2021. White (HC0) t-statistics in parentheses.",
           label_width=1.2)

Linear asset pricing model, Traded=True  (T=504, N=17, K=6)
        lambda  t-stat
Mkt-RF   0.743   3.732
SMB      0.095   0.736
HML      0.203   1.503
RMW      0.376   3.522
CMA      0.251   2.867
Mom      0.529   2.651
Cross-sectional R2: -2.365
J-test: J = 38.27, p-value = 0.0023, chi2(17)
Factor loadings (t-stats in parentheses):
            Mkt-RF           SMB           HML           RMW           CMA           Mom
Food   0.79 (21.7)  -0.10 (-1.8)  -0.10 (-1.8)    0.55 (8.0)    0.46 (5.0)    0.02 (0.5)
Mines  1.04 (13.3)    0.43 (3.7)  -0.01 (-0.1)    0.04 (0.2)    0.53 (2.3)  -0.00 (-0.1)
Oil    1.02 (15.7)    0.13 (1.3)    0.40 (2.7)    0.25 (2.1)    0.24 (1.2)    0.02 (0.3)
Clths  1.05 (22.7)    0.48 (6.7)    0.10 (1.2)    0.70 (8.3)  -0.05 (-0.4)  -0.22 (-5.4)
Durbl  1.08 (30.0)    0.38 (7.5)    0.04 (0.5)    0.36 (5.1)    0.30 (3.1)  -0.19 (-4.1)
Chems  1.15 (30.3)    0.12 (2.5)    0.19 (2.6)    0.37 (5.5)    0.20 (1.8)  -0.13 (-2.9)
Cnsum  0.81 (24.0)  -0.30 (-5.2)  -0.31 (

,"λ, traded","t, traded","λ, non-traded","t, non-traded"
Mkt-RF,0.743,3.732,0.814,35.654
SMB,0.095,0.736,-0.523,-4.900
HML,0.203,1.503,-0.329,-5.257
RMW,0.376,3.522,0.395,6.457
CMA,0.251,2.867,-0.196,-2.140
Mom,0.529,2.651,0.353,1.101


Panel B: Model fit and J-test


,"λ, traded","t, traded","λ, non-traded","t, non-traded"
Cross-sectional R²,-2.365,–,0.693,–
J statistic,38.273,–,4.314,–
p-value,0.002,–,0.960,–
Degrees of freedom,17,–,11,–


Table 1.5 - Factor loadings of the 17 industry portfolios (time-series pass)


,Mkt-RF,SMB,HML,RMW,CMA,Mom
Food,0.79 (21.7),-0.10 (-1.8),-0.10 (-1.8),0.55 (8.0),0.46 (5.0),0.02 (0.5)
Mines,1.04 (13.3),0.43 (3.7),-0.01 (-0.1),0.04 (0.2),0.53 (2.3),-0.00 (-0.1)
Oil,1.02 (15.7),0.13 (1.3),0.40 (2.7),0.25 (2.1),0.24 (1.2),0.02 (0.3)
Clths,1.05 (22.7),0.48 (6.7),0.10 (1.2),0.70 (8.3),-0.05 (-0.4),-0.22 (-5.4)
Durbl,1.08 (30.0),0.38 (7.5),0.04 (0.5),0.36 (5.1),0.30 (3.1),-0.19 (-4.1)
Chems,1.15 (30.3),0.12 (2.5),0.19 (2.6),0.37 (5.5),0.20 (1.8),-0.13 (-2.9)
Cnsum,0.81 (24.0),-0.30 (-5.2),-0.31 (-4.9),0.37 (4.8),0.55 (5.3),-0.00 (-0.1)
Cnstr,1.20 (34.7),0.27 (5.2),0.15 (2.3),0.36 (4.8),0.09 (0.8),0.02 (0.4)
Steel,1.33 (23.6),0.49 (5.7),0.21 (1.9),-0.21 (-1.8),0.16 (0.9),-0.19 (-3.3)
FabPr,1.03 (29.2),0.42 (7.2),0.02 (0.3),0.54 (6.4),0.10 (1.1),-0.15 (-4.0)


## 1d. Interpretation [10 pts]
Answered in the write-up (2–3 sentences each): significant risk premia under `Traded=True`; J-test conclusion; why the J-test null distribution depends on `Traded`; which version is theoretically appropriate here.

# Problem 2 – Maximum Likelihood Estimation [35 pts]
Dependent variable: Mkt-RF. Regressors: SMB, HML, RMW, CMA, Mom (same 1980–2021 monthly data as Problem 1).

## 2a. `Q2_OLSStudentT()` [8 pts]
Inputs: `df` (first column = dependent variable, others = regressors), integer `dof > 0`. (i) OLS on a constant and the regressors; (ii) residuals $\hat e_t$ and $\hat\sigma$; (iii) log-likelihood assuming $\hat e_t/\hat\sigma$ i.i.d. $t(\text{dof})$. Returns `llf`. Apply with dof = 5 and dof = 30; explain the difference (tail behaviour of the residuals).

In [13]:
def Q2_OLSStudentT(df, dof):
    """
    OLS of df's first column on a constant + the other columns, then the
    log-likelihood assuming the standardized residuals e_t / sigma are i.i.d. t(dof).
    """
    if not (isinstance(dof, (int, np.integer)) and dof > 0):
        raise ValueError("dof must be an integer > 0")

    # (i) OLS of the first column on a constant and the other columns
    y = df.iloc[:, 0]
    X = sm.add_constant(df.iloc[:, 1:])
    res = sm.OLS(y, X, missing="drop").fit()

    # (ii) residuals and their estimated standard deviation
    e = res.resid
    sigma = np.sqrt(res.scale)          # sqrt(SSR / (T - k))

    # (iii) log-likelihood with z_t = e_t / sigma ~ i.i.d. t(dof)
    z = e / sigma
    T = len(e)
    llf = np.sum(stats.t.logpdf(z, df=dof)) - T * np.log(sigma)

    return llf

In [14]:
q2_df = factors[["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]].dropna()   # Mkt-RF first = dependent variable

llf_5  = Q2_OLSStudentT(q2_df, 5)
llf_30 = Q2_OLSStudentT(q2_df, 30)

print(f"Log-likelihood, t(5):  {llf_5:.2f}")
print(f"Log-likelihood, t(30): {llf_30:.2f}")

Log-likelihood, t(5):  -1407.33
Log-likelihood, t(30): -1400.47


In [15]:
def llf_std_t(df, dof):
    # unit-variance Student t (requires dof > 2)
    y = df.iloc[:, 0]
    X = sm.add_constant(df.iloc[:, 1:])
    res = sm.OLS(y, X, missing="drop").fit()
    e = res.resid
    sigma = np.sqrt(res.scale)
    z = e / sigma
    c = np.sqrt(dof / (dof - 2))              # rescales t(dof) to variance 1
    T = len(e)
    return np.sum(stats.t.logpdf(z * c, df=dof)) + T * np.log(c) - T * np.log(sigma)

print("Standardized t(5): ", round(llf_std_t(q2_df, 5), 2))
print("Standardized t(30):", round(llf_std_t(q2_df, 30), 2))
print("Normal (OLS llf):  ", round(sm.OLS(q2_df.iloc[:, 0], sm.add_constant(q2_df.iloc[:, 1:])).fit().llf, 2))

Standardized t(5):  -1389.95
Standardized t(30): -1399.08
Normal (OLS llf):   -1405.75


## 2b. `Q2_LRTest()` [9 pts]
Inputs: `df`, integer `dof > 0`. Uses `Q2_OLSStudentT()` to compute the LR statistic for $H_0$: all coefficients except the constant are zero. Returns the LR statistic and its p-value. Apply with dof = 5: (i) restricted model and its log-likelihood vs. unrestricted; (ii) do the five factors jointly predict Mkt-RF?; (iii) does the LR statistic change with dof = 30?

## 2c. `Q2_ProfileLikelihood()` [9 pts]
Input: `df`. Evaluates `Q2_OLSStudentT(df, dof)` at $\text{dof}_j = e^{j/12}$, $j = 0, \dots, 50$, plots the profile likelihood and returns a 51×2 array [dof, llf]. Interpret: (i) shape of the curve; (ii) location of the maximum and implied MLE of dof.

## 2d. MLE of the degrees of freedom [9 pts]
Answered in the write-up from the profile likelihood in 2c: (i) the estimate (≈ 1, ≈ 5, ≈ 15 or > 60); (ii) tail behaviour; (iii) ruling out Cauchy (ν = 1) and near-normal (ν > 60); (iv) consequences for the inference in Problem 1.

# Problem 3 – PCA and Factor Models [15 pts]
No coding required. Answered in the write-up: (a) recent news article and citation; (b) falsifiable hypothesis; (c) PCA design (matrix X, demeaning/standardizing, choice of K); (d) mathematical model ($\Sigma = V\Lambda V'$, $X \approx F V_K'$); (e) conversion to a linear factor model in the spirit of `Q1_AssetPricingModel()`.